### Final Data Cleaning

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_parquet("009_weather_included.parquet").copy()
print('Number of people involved: ', df.shape[0],
      '\nNumber of variables:          ', df.shape[1],
      '\nNumber of accidents:       ', df['Protocollo'].nunique())

Number of people involved:  19416 
Number of variables:           67 
Number of accidents:        17954


### Latitude & Longitude Missing

In [3]:
df[df['Latitude'].isna() | df['Longitude'].isna()][['Gruppo', 'Localizzazione1',
                                                    'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
                                                    'DaSpecificare']]

,Gruppo,Localizzazione1,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare
0,8,Statale entro l'abitato,VIA CASILINA,da specificare,,,,altezza civico 1800
1,1,Strada Urbana,PIAZZA DEI CINQUECENTO,da specificare,,,,.
2,1,Strada Urbana,VIA EMILIA,in corrispondenza,,del civico n.,104,
3,1,Strada Urbana,VIA EMILIA,in corrispondenza,,del civico n.,104,
4,13,Strada Urbana,VIALE VASCO DE GAMA,all'intersezione semaforizzata con,VIA DELLE SIRENE,VIA DELLE SIRENE,,
...,...,...,...,...,...,...,...,...
19168,1,Strada Urbana,VIA DEL TRITONE,da specificare,,,,altezza via Poli
19190,1,Strada Urbana,PIAZZA DEI CINQUECENTO,in prossimitÃ,,del palo luce n.,103,
19194,1,Strada Urbana,VIA DEL CORSO,in corrispondenza,,del civico n.,241,
19269,20,Regionale entro l'abitato,VIA CASSIA,in prossimitÃ,,del civico n.,902,


For the addresses where the latitude and longitude are missing, we will make an Excel file containing the protocols and location information. The file will be used to clean the addresses so that we can geocode the missing locations using a Google Developers / Maps API. 

In [4]:
# pip install openpyxl

In [27]:
'''
mask = df['Latitude'].isna() | df['Longitude'].isna()
cols = ['Protocollo', 'Gruppo', 'Localizzazione1', 'STRADA1', 'Localizzazione2',
        'STRADA2', 'Strada02', 'Chilometrica', 'DaSpecificare']

out = df.loc[mask, cols].copy()

# Save in current working directory
out.to_excel("missing_latitude_or_longitude.xlsx", index=False)  # creates missing_latlon.xlsx
'''

'\nmask = df[\'Latitude\'].isna() | df[\'Longitude\'].isna()\ncols = [\'Protocollo\', \'Gruppo\', \'Localizzazione1\', \'STRADA1\', \'Localizzazione2\',\n        \'STRADA2\', \'Strada02\', \'Chilometrica\', \'DaSpecificare\']\n\nout = df.loc[mask, cols].copy()\n\n# Save in current working directory\nout.to_excel("missing_latitude_or_longitude.xlsx", index=False)  # creates missing_latlon.xlsx\n'

To find the latitudes and longitudes for the accident locations identified by a street name and the 'luce palo' number, I used the app Illumina Roma and searched for them, one by one (700+ locations)

Where an address or two streets were given, to find the Latitudes & Longitudes, I activated a geocoding Google Developers API key
https://developers.google.com/maps/documentation/geocoding/get-api-key?setupProd=enable

In Excel, I used the information in columns: STRADA1, Localizzazione2, STRADA2, Strada02, Chilometrica, and DaSpecificare and from these made a spreadsheet of the addresses required: 'Street, civic number, Roma, Italia' OR 'Street A & Street B' (for intersections). 

Other information was also gleaned from the address information: intersections indicates the accident happened at an intersection, traffic lights/pedestrian crossings indicate there were roadmarkings_onroad and roadmarkings_signs. 

Protocols where latitude and longitude impossible to locate: 9 values
The street name was given, but there were no further indications regarding the position of the accident (civic number, lamppost number, etc). These protocols will be deleted.

Protocols where a pedestrian crossing was present, given the description in the DaSpecificare column: 4 values
For these, we set:
- 'road_markings_onroad' = 1 
- 'road_markings_signposts' = 1

Protocols where traffic lights were present, given the description: 162 values
- 'road_markings_onroad' = 1 
- 'road_markings_signposts' = 1

Protocols where intersection present, given the description: 704 values
- check 'road_features' to see if 'intersection' was noted.

Then we will load a csv with the list of protocols, with the latitudes and longitudes found using the geocoding API, then join these to our dataframe. 

In [5]:
# Protocols for the accidents where it is impossible to deduce the latitude and longitude:
protocols_impossible_location = [
    1951018, 1938726, 3955350, 1931666, 1947662, 1942676, 1952972, 3975062, 3709985]

# Protocols where a pedestrian crossing is present:
protocols_ped_crossing = [1966000, 1931849, 5758507, 5685812]

# Protocols where traffic lights are present:
protocols_traffic_lights_present = [1939097, 1957933, 1938330, 1937935, 1949871, 1946442, 1945351, 4089136, 5807662, 5807662, 1936986, 1951932, 1952513, 1944493, 3903669, 1955534, 1946647, 1956312, 1939463, 1960415, 1958159, 1953227, 1958523, 1937174, 1951865, 1938590, 1959543, 1960472, 1954365, 1933823, 1947909, 1974593, 1931793, 1945261, 1937650, 1939473, 1947468, 1945072, 1945072, 1951682, 1946610, 1950824, 1981662, 1935296, 1943618, 1936389, 4595794, 3964767, 1931691, 1945591, 1941073, 1957301, 1940615, 1959909, 1933244, 1946086, 1953695, 1937140, 1959889, 1932644, 1961040, 1950567, 4120140, 4006282, 1955146, 1958675, 1945774, 1972543, 1957059, 1941541, 1977908, 1937555, 1938994, 1938994, 1949537,
                                    1949537, 1945156, 1935965, 1946877, 1946877, 1939264, 1955295, 1942304, 1962106, 1940776, 1946727, 1952773, 1949349, 4667955, 1952470, 1935679, 1962081, 1949249, 1958288, 1931975, 3901877, 3901877, 1945897, 1946070, 1946070, 1946070, 1956178, 1963125, 1941745, 3825949, 1940408, 1958356, 1959219, 1945700, 1945700, 1945760, 1943901, 1933168, 1950593, 1958218, 1932705, 1946245, 1953161, 1962819, 1941953, 1944301, 1953002, 1954155, 1947924, 1958075, 1933027, 1933027, 1933027, 2659597, 1960967, 4670663, 1937692, 1950435, 1952058, 1960286, 1944325, 1935255, 1946203, 1933416, 1955705, 1957314, 1957314, 1937788, 1950272, 1948235, 1949474, 3728706, 1947061, 1940735, 1948030, 3999551, 3999551, 1945192]

# Protocols where there is an intersection:
protocols_intersection_present = [1966000, 1931849, 5758507, 5685812, 1939097, 1957933, 1938330, 1937935, 1949871, 1946442, 1945351, 4089136, 5807662, 5807662, 1936986, 1951932, 1952513, 1944493, 3903669, 1955534, 1946647, 1956312, 1939463, 1960415, 1958159, 1953227, 1958523, 1937174, 1951865, 1938590, 1959543, 1960472, 1954365, 1933823, 1947909, 1974593, 1931793, 1945261, 1937650, 1939473, 1947468, 1945072, 1945072, 1951682, 1946610, 1950824, 1981662, 1935296, 1943618, 1936389, 4595794, 3964767, 1931691, 1945591, 1941073, 1957301, 1940615, 1959909, 1933244, 1946086, 1953695, 1937140, 1959889, 1932644, 1961040, 1950567, 4120140, 4006282, 1955146, 1958675, 1945774, 1972543, 1957059, 1941541, 1977908, 1937555, 1938994, 1938994, 1949537, 1949537, 1945156, 1935965, 1946877, 1946877, 1939264, 1955295, 1942304, 1962106, 1940776, 1946727, 1952773, 1949349, 4667955, 1952470, 1935679, 1962081, 1949249, 1958288, 1931975, 3901877, 3901877, 1945897, 1946070, 1946070, 1946070, 1956178, 1963125, 1941745, 3825949, 1940408, 1958356, 1959219, 1945700, 1945700, 1945760, 1943901, 1933168, 1950593, 1958218, 1932705, 1946245, 1953161, 1962819, 1941953, 1944301, 1953002, 1954155, 1947924, 1958075, 1933027, 1933027, 1933027, 2659597, 1960967, 4670663, 1937692, 1950435, 1952058, 1960286, 1944325, 1935255, 1946203, 1933416, 1955705, 1957314, 1957314, 1937788, 1950272, 1948235, 1949474, 3728706, 1947061, 1940735, 1948030, 3999551, 3999551, 1945192, 1954614, 1939852, 1951903, 3807620, 1947810, 1937784, 1945969, 1935039, 1939411, 1954480, 1935955, 1935955, 1958006, 4366321, 1955948, 1951799, 1945188, 1944224, 1943699, 1934809, 1945214, 1945214, 1945214, 1945214, 1957448, 1938437, 1938446, 1953461, 1936379, 1939587, 1945646, 1963319, 1940575, 1957057, 1957249, 1957249, 1935077, 1935077, 1955242, 1937567, 1947766, 1941644, 1945167, 1937049, 1955882, 1950580, 1933095, 1946566, 1963749, 1959541, 1936612, 1945679, 1956284, 1940765, 1939701, 1936287, 1943391, 1942748, 1960124, 1942030, 1960765, 1938630, 1947914, 1941268, 1971597, 1955817, 1931991, 1931991, 5183399, 3803499, 1936431, 1960573, 1938626, 1954511, 1953790, 1953790, 1952074, 1952074, 1939335, 1940645, 1945866, 1934625, 1946461, 1946461, 1960850, 1990646, 1956758, 1939309, 1935344, 1961229, 1959300, 1935725, 1935725, 1946738, 1933888, 1945607, 1934906, 1951993, 1940294, 5716974, 1933722, 1948832, 3979006, 1956169, 1936580, 1952802, 1952802, 1955198, 1944218, 1932200, 1943181, 5947989, 1954761, 1954666, 1935209, 1959568, 1939261, 1936738, 1949925, 1934341, 1948321, 1949207, 1961394, 1938579, 1938815, 1957554, 1947665, 1956182, 1936028, 1946736, 1951328, 1954713, 1950920, 1949164, 1935951, 1957108, 1935530, 1948804, 1950223, 1962306, 1939046, 1939312, 1959900, 1934464, 1956936, 1962394, 1952604, 1945705, 1961288, 1953106, 1958012, 1961294, 1947149, 1939107, 5454444, 1945909, 1935676, 1946341, 4942703, 1938787, 1962896, 1962896, 1954154, 1934896, 1935246, 1954336, 1959623, 1946311, 1946987, 1940174, 1945162, 1951598, 1946697, 1945561, 1945561, 1932051, 1954563, 1939736, 1948780, 1938494, 1953507, 1953099, 1953920, 1932864, 1955824, 1955824, 5145752, 2630615, 2630615, 1935673, 1957656, 4366257, 1957699,
                                  1940866, 1940866, 1957775, 3952427, 4152232, 1956529, 1938122, 1953927, 1934503, 1934503, 1935230, 1960601, 1939666, 1939874, 1932286, 1947744, 1934256, 5958771, 1962508, 4256591, 4254716, 4254716, 1956432, 1956432, 1942775, 1945014, 1940189, 1957929, 1933842, 1958204, 1935061, 1956850, 1934887, 1941267, 1958115, 1951608, 1951608, 4349486, 1932763, 1951536, 1958780, 1961242, 1934644, 1955004, 1952201, 1945469, 1955604, 1942330, 1950356, 1976346, 1931739, 1938382, 3725598, 1933200, 1938550, 1948261, 1948261, 1969403, 1948067, 1935063, 1957916, 1959217, 1941817, 1941817, 1941817, 1945971, 1937975, 1948395, 1990409, 1939168, 1940340, 1940340, 1940340, 4108516, 1953782, 1947452, 1951369, 1936175, 1946477, 1980032, 1959822, 1932512, 1932373, 4529616, 5036433, 1964409, 1952669, 1943675, 1960972, 1961338, 1961338, 1936656, 1954862, 1932663, 1956361, 1943860, 5712979, 1941786, 1938234, 1958253, 1958253, 1952769, 1939903, 1958193, 1947229, 1939959, 1934623, 1945497, 1940548, 1939702, 2596191, 1950922, 1936208, 1940614, 1945092, 1945092, 3889881, 3889881, 3946766, 1942744, 1942916, 1936544, 1946418, 1946418, 1968766, 1933503, 1947655, 1936052, 1961710, 1959201, 1957056, 1962849, 1943337, 1944363, 1942860, 1936615, 1950177, 1948426, 1944225, 1959239, 1959239, 1944508, 1948269, 1956279, 1951654, 4827788, 1961308, 1956874, 1939351, 1947064, 1939958, 1947832, 1953562, 1947845, 1936292, 1943258, 1954899, 4088589, 1942781, 1985578, 1934009, 1933709, 1953188, 1965775, 1957105, 1939137, 1932346, 1960759, 1965927, 1952858, 1963186, 1963186, 1951967, 1959598, 1973057, 1961950, 1936311, 1936311, 1943226, 1932315, 1948483, 1967941, 1961120, 5392060, 1958240, 1952770, 3687805, 3687805, 6084276, 1936630, 1942833, 1971390, 1971390, 1950662, 1949664, 1960797, 3657994, 4355021, 1951721, 1950097, 1937998, 5932403, 1955061, 1947805, 1955100, 1949458, 1944555, 1952332, 1933169, 1951894, 1937822, 4990788, 1947006, 1961389, 1939656, 1961056, 1956246, 1939422, 3981191, 1959652, 1953063, 1969012, 1960757, 1945942, 1945942, 1959441, 1945604, 1954675, 1954675, 1944545, 1932609, 1931911, 1933856, 1955989, 1933476, 1940090, 1942063, 1954325, 1955759, 6028636, 4001445, 1932427, 1957536, 1954390, 1931864, 1950634, 1956296, 1948985, 3680335, 1959265, 1934928, 1961479, 1959928, 1941060, 1955050, 3796270, 1934533, 1951432, 1953264, 1942269, 1942269, 1943912, 1940348, 1946511, 3934971, 1957092, 1954127, 1942106, 1942106, 1942849, 1961588, 1950487, 2240786, 1941564, 5847836, 1992614, 1962883, 1956766, 1960780, 1954068, 1953134, 1931929, 1952574, 1946523, 1953755, 3761529, 1934598, 1952214, 1937950, 1963032, 1962389, 1942083, 1938356, 1944461, 1951317, 1945651, 1945651, 1943596, 1934845, 1953640, 1937579, 1959317, 1951534, 1960012, 1946160, 1967379, 1936200, 1946771, 1954334, 1958009, 1958590, 1937809, 1952646, 1952646, 1937078, 1949484, 1957470, 1935166, 1959473, 1952351, 5962959, 1947278, 1941991, 1952790, 1932580, 1936037, 1932809, 1938376, 1941596, 1946757, 1959226, 1959996, 3188655, 5628583, 1953690, 1937779, 3144978, 3144978, 1959916, 1957060, 1949012, 1936857, 1955495, 1952268, 1951983, 1954524, 1947944, 1938297, 1950723, 1958215, 1941340, 1937803, 1936730, 1961420]

Drop impossible to locate accidents:

In [6]:
# Specify protocols to drop
to_drop = df["Protocollo"].isin(protocols_impossible_location)

# Delete and reset index
df = df.loc[~to_drop].reset_index(drop=True)

Check road markings values for accidents where there is a pedestrian crossing:

In [7]:
# For the list of protocols where there is a pedestrian crossing, we check the 'road_markings_onroad' and 'road_markings_signposts' values:
df.loc[
    df['Protocollo'].isin(protocols_ped_crossing),
    ['Protocollo', 'road_markings_onroad', 'road_markings_signposts', 'road_markings_traffic_lights']]

,Protocollo,road_markings_onroad,road_markings_signposts,road_markings_traffic_lights
16,1931849,1,1,0
2167,1966000,1,1,0
17638,5685812,1,1,0
18076,5758507,1,1,0


The onroad and signpost road markings are already noted for these protocols. 

Check values for accidents where traffic lights are present:

In [8]:
df.loc[df["Protocollo"].isin(protocols_traffic_lights_present),
       "road_markings_traffic_lights"].value_counts(dropna=False)

road_markings_traffic_lights
1    121
0     32
Name: count, dtype: int64

In [9]:
df.loc[
    df["Protocollo"].isin(protocols_traffic_lights_present)
    & (df["road_markings_traffic_lights"] == 0),
    ["Protocollo",
     "road_markings_onroad",
     "road_markings_signposts",
     "road_markings_traffic_lights",
     "road_markings_temporary",
     "road_features"]
].sort_values("Protocollo").reset_index(drop=True)

,Protocollo,road_markings_onroad,road_markings_signposts,road_markings_traffic_lights,road_markings_temporary,road_features
0,1931793,1,1,0,0,Intersection
1,1933027,1,1,0,0,Intersection
2,1933027,1,1,0,0,Intersection
3,1933027,1,1,0,0,Intersection
4,1933416,1,1,0,0,Intersection
5,1935965,1,1,0,0,Intersection
6,1936986,1,1,0,0,Intersection
7,1940408,1,1,0,0,Intersection
8,1941541,1,1,0,0,Intersection
9,1944301,0,0,0,1,Intersection


In [10]:
# Change the 'road_markings_traffic_lights' and 'road_markings_onroad' values:
mask = df['Protocollo'].isin(protocols_traffic_lights_present)
df.loc[mask & (df['road_markings_traffic_lights'] == 0),
       'road_markings_traffic_lights'] = 1
df.loc[mask & (df['road_markings_traffic_lights'] == 0) & (
    df['road_markings_temporary'] == 1), 'road_markings_onroad'] = 1

In [11]:
# Check the values have been changed:
df.loc[
    df["Protocollo"].isin(protocols_traffic_lights_present)
    & (df["road_markings_traffic_lights"] == 0),
    ["Protocollo",
     "road_markings_onroad",
     "road_markings_signposts",
     "road_markings_traffic_lights",
     "road_markings_temporary",
     "road_features"]
].sort_values("Protocollo").reset_index(drop=True)

,Protocollo,road_markings_onroad,road_markings_signposts,road_markings_traffic_lights,road_markings_temporary,road_features


In [12]:
df['road_features'].value_counts()

road_features
Straight road         12953
Intersection           5053
Curve                   820
Roundabout              336
Slope                   182
road_feature_other       63
Name: count, dtype: int64

Check protocols where an intersection should be featured, based on the address (two road names given or 'intersection' specifically mentioned in the address description):

In [13]:
df.loc[
    df['Protocollo'].isin(protocols_intersection_present),
    ['road_features']].value_counts()

road_features     
Intersection          496
Straight road         185
Curve                  11
Roundabout             11
Slope                   1
road_feature_other      0
Name: count, dtype: int64

In order to conserve the information: roundabout, curve, slope - we will leave these values untouched as replacing them with 'intersection' would lose the information provided. 

Where the road feature 'straight road' was noted, we will replace this with 'intersection' as 'Straight road' was most likely selected by mistake.

In [14]:
mask_int = df['Protocollo'].isin(protocols_intersection_present)
df.loc[mask_int & (df['road_features'] == 'Straight road'),
       'road_features'] = 'Intersection'

In [15]:
# Double-check protocols where an intersection was noted
df.loc[
    df['Protocollo'].isin(protocols_intersection_present),
    ['road_features']].value_counts()

road_features     
Intersection          681
Curve                  11
Roundabout             11
Slope                   1
Straight road           0
road_feature_other      0
Name: count, dtype: int64

Now we will integrate the missing Latitudes and Longitudes from the excel sheet:

In [16]:
# --- 1) Read Excel → df_positions (one row per Protocollo, numeric Lat/Lon) ---
df_positions = (
    pd.read_excel(
        r"C:\Users\lucyq\Dropbox\AMDP\THESIS\Lat_long_found.xlsx",
        dtype={"Protocollo": "Int64"}
    )[["Protocollo", "Latitude", "Longitude"]]
    .assign(
        Latitude=lambda d: pd.to_numeric(d["Latitude"], errors="coerce"),
        Longitude=lambda d: pd.to_numeric(d["Longitude"], errors="coerce"),
    )
    .dropna(subset=["Protocollo"])  # keep only rows with a valid Protocollo
    # one donor per Protocollo
    .drop_duplicates(subset=["Protocollo"], keep="last")
)

In [17]:
# --- 2) Prep your main df and align types (do NOT set index) ---
df = df.copy()
df["Protocollo"] = pd.to_numeric(
    df["Protocollo"], errors="coerce").astype("Int64")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

In [18]:
df_positions.head(3)

,Protocollo,Latitude,Longitude
0,1966000,41.738369,12.279595
1,1931849,41.863988,12.671015
2,5758507,41.937314,12.469957


In [19]:
# --- 3) Merge donors (suffix _donor) ---
merged = df.merge(
    df_positions,
    on="Protocollo",
    how="left",
    suffixes=("", "_donor")
)

# --- 4) Identify rows needing fill (one OR both missing) and with a valid donor (both present) ---
need_fill = merged["Latitude"].isna() | merged["Longitude"].isna()
has_donor = merged["Latitude_donor"].notna(
) & merged["Longitude_donor"].notna()
ix = need_fill & has_donor

# --- 5) Overwrite BOTH Latitude & Longitude from donor where needed ---
merged.loc[ix, "Latitude"] = merged.loc[ix, "Latitude_donor"]
merged.loc[ix, "Longitude"] = merged.loc[ix, "Longitude_donor"]

# --- 6) Drop donor cols and keep result as df ---
merged = merged.drop(columns=["Latitude_donor", "Longitude_donor"])
df = merged

# --- 7) Optional diagnostics ---
exactly_one_missing_before = (need_fill & ~(
    merged["Latitude"].isna() & merged["Longitude"].isna())).sum()
only_lat_missing_before = (
    need_fill & merged["Latitude"].isna() & merged["Longitude"].notna()).sum()
only_lon_missing_before = (
    need_fill & merged["Longitude"].isna() & merged["Latitude"].notna()).sum()

print("Rows needing fill (before):", int(need_fill.sum()))
print("Rows overwritten (had donor):", int(ix.sum()))

remaining_missing_after = df[["Latitude",
                              "Longitude"]].isna().any(axis=1).sum()

print("\nRows still missing after overwrite:", int(remaining_missing_after))

Rows needing fill (before): 2172
Rows overwritten (had donor): 2172

Rows still missing after overwrite: 0


In [20]:
print(df[["Latitude", "Longitude"]].describe())

           Latitude     Longitude
count  1.940700e+04  1.940700e+04
mean   2.155599e+05  6.496160e+04
std    3.002360e+07  9.047996e+06
min    3.695157e+01  1.187785e+01
25%    4.186736e+01  1.245230e+01
50%    4.189431e+01  1.248907e+01
75%    4.191559e+01  1.253579e+01
max    4.182559e+09  1.260467e+09


So the max and min Latitudes and Longitudes still look wrong. 

In [21]:
LAT_MIN, LAT_MAX = 41.6, 42.1
LON_MIN, LON_MAX = 12.1, 12.8

# rows outside Rome bbox OR physically impossible coords
mask_bad = (
    (df["Latitude"].abs() > 90) | (df["Longitude"].abs() > 180) |
    (df["Latitude"] < LAT_MIN) | (df["Latitude"] > LAT_MAX) |
    (df["Longitude"] < LON_MIN) | (df["Longitude"] > LON_MAX)
)

# list of Protocollo with issues
bad_protocols = (
    df.loc[mask_bad, "Protocollo"]
      .dropna()
      .drop_duplicates()
      .sort_values()
)

print("Rows with weird coords:", int(mask_bad.sum()))
print("Distinct Protocollo affected:", int(bad_protocols.nunique()))
print(bad_protocols.tolist()[:25])   # preview first 25

# If you want to inspect the actual rows:
bad_rows = df.loc[mask_bad, ["Protocollo", "Latitude",
                             "Longitude", "STRADA1", "Localizzazione2", "STRADA2", "Strada02", "Chilometrica",
                             "DaSpecificare"]].sort_values("Protocollo")
bad_rows.head(10)

Rows with weird coords: 10
Distinct Protocollo affected: 9
[1934687, 1936311, 1945037, 1948978, 1949921, 1951365, 1955410, 4152232, 5179368]


,Protocollo,Latitude,Longitude,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare
208,1934687,4.173880e+01,1.295965e+01,VIA S. TOMMASO D'AQUINO,in corrispondenza,,del civico n.,9,
328,1936311,3.695157e+01,1.453568e+01,VIA PRINCIPE UMBERTO,all'intersezione con,VIA LAMARMORA,VIA LAMARMORA,,
329,1936311,3.695157e+01,1.453568e+01,VIA PRINCIPE UMBERTO,all'intersezione con,VIA LAMARMORA,VIA LAMARMORA,,
861,1945037,4.192723e+01,1.309122e+01,VIA POGGIO VERDE,da specificare,,,,in prossimitÃ del fronte civico 4
1136,1948978,4.173698e+01,1.295799e+01,VIA S. TOMMASO D'AQUINO,in corrispondenza,,del civico n.,4,
1190,1949921,4.203863e+01,1.187785e+01,VIA DELLE CAMELIE,in corrispondenza,,del civico n.,2,
1267,1951365,4.182559e+09,1.260314e+01,VIA ANAGNINA,da specificare,,,,dir. Rm altezza ikea fronte palo IP n. 8
1531,1955410,4.195795e+01,1.260467e+09,VIA RINALDO D'AQUINO,da specificare,,,,altezza palo acea n. 22
12433,4152232,4.215546e+01,1.268072e+01,VIA SALARIA,da specificare,,,,"via Cortona , rampa dir. Rieti"
15957,5179368,4.148098e+01,1.248098e+01,PIAZZA DELL'ARA COELI,in prossimitÃ,,del palo luce n.,7,


Correcting these 'by hand' (using the app Roma Illumina and Google maps) we have:
NB numbers given to 6 dp to align with most other numbers in the dataset.

In [22]:
df.loc[df["Protocollo"] == 1934687, ["Latitude", "Longitude"]] = [
    41.911061, 12.447765]  # Similar address Latina
df.loc[df["Protocollo"] == 1936311, ["Latitude", "Longitude"]] = [
    41.895455, 12.507458]  # Similar address Sicily
df.loc[df["Protocollo"] == 1945037, ["Latitude", "Longitude"]] = [
    41.856255, 12.417005]  # Similar address Subiaco
df.loc[df["Protocollo"] == 1948978, ["Latitude", "Longitude"]] = [
    41.910509, 12.447752]
df.loc[df["Protocollo"] == 1949921, ["Latitude", "Longitude"]] = [
    41.877269, 12.566811]
df.loc[df["Protocollo"] == 1955410, [
    "Latitude", "Longitude"]] = [41.957946, 12.604673]
df.loc[df["Protocollo"] == 4152232, ["Latitude", "Longitude"]] = [
    41.965590, 12.506962]
df.loc[df["Protocollo"] == 5179368, [
    "Latitude", "Longitude"]] = [41.895036, 12.481214]

df = df[df["Protocollo"] != 1951365].reset_index(drop=True)

In [23]:
print(df[["Latitude", "Longitude"]].describe())

           Latitude     Longitude
count  19406.000000  19406.000000
mean      41.888286     12.491736
std        0.049694      0.073991
min       41.634360     12.239180
25%       41.867370     12.452238
50%       41.894320     12.489050
75%       41.915582     12.535770
max       42.095470     12.789310


### ORGANISE COLUMNS

In [24]:
df.columns

Index(['Protocollo', 'DataOraIncidente', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'parked_vehicle_involved',
       'driver_injury', 'driver_gender', 'passenger_no_of_females',
       'passenger_no_of_males', 'passenger_no_of_unknown_sex',
       'passengers_killed', 'passengers_uninjured', 'passengers_injured',
       'phase_of_day', 'YEAR', 'MONTH', 'DATE', 'TIME', 'DAY',
       'pedestrian_gender', 'road_conditions', 'road_markings_absent',
       'road_markings_onroad', 'road_markings_signposts',
       'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type',
       'TipoStradaDifficulty', 'visibility', 'pedestrian_injury',
       'lighting_insufficient', 'traffic_density', 'vehicle_type',
       'vehicle_unknown', 'hit_and_run', 'DataOraIncidente_utc_rounded',
       'time', 'temperature_2m (°C)', 'relative_humi

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19406 entries, 0 to 19405
Data columns (total 67 columns):
 #   Column                            Non-Null Count  Dtype                      
---  ------                            --------------  -----                      
 0   Protocollo                        19406 non-null  Int64                      
 1   DataOraIncidente                  19406 non-null  datetime64[ns, Europe/Rome]
 2   Gruppo                            19406 non-null  Int64                      
 3   Localizzazione1                   19406 non-null  object                     
 4   STRADA1                           19406 non-null  object                     
 5   Localizzazione2                   19406 non-null  object                     
 6   STRADA2                           19406 non-null  object                     
 7   Strada02                          19406 non-null  object                     
 8   Chilometrica                      19406 non-null  object

In [26]:
df[df['weather_days_since_last_rain'].isna()]

,Protocollo,DataOraIncidente,Gruppo,Localizzazione1,STRADA1,Localizzazione2,STRADA2,Strada02,Chilometrica,DaSpecificare,...,wind_speed_10m (km/h),wind_gusts_10m (km/h),time_local,time_utc,weather_condition_noted,weather_wet,weather_snowy,weather_icy,weather_days_since_last_rain,weather_cumulative_rain_past_24h
0,1931584,2013-01-01 19:25:00+01:00,8,Statale entro l'abitato,VIA CASILINA,da specificare,,,,altezza civico 1800,...,7.2,11.5,2013-01-01 19:00:00+01:00,2013-01-01 18:00:00+00:00,Rain,0,0,0,NaN,0.0
1,1931673,2013-01-01 02:15:00+01:00,1,Strada Urbana,VIA EMILIA,in corrispondenza,,del civico n.,104,,...,4.6,9.0,2013-01-01 02:00:00+01:00,2013-01-01 01:00:00+00:00,Clear,0,0,0,NaN,0.0
2,1931673,2013-01-01 02:15:00+01:00,1,Strada Urbana,VIA EMILIA,in corrispondenza,,del civico n.,104,,...,4.6,9.0,2013-01-01 02:00:00+01:00,2013-01-01 01:00:00+00:00,Clear,0,0,0,NaN,0.0
42,1932052,2013-01-01 08:20:00+01:00,10,Strada Urbana,VIA DI TOR VERGATA,in prossimitÃ,,del civico n.,381,,...,5.6,9.7,2013-01-01 08:00:00+01:00,2013-01-01 07:00:00+00:00,Clear,0,0,0,NaN,0.0


As these are accidents on the first day, only 4 rows are affected if we impute a value of 5 days since the last rainfall, rather than deleting these rows. (Open meteo reports rainfall on 27/12/2012)

In [27]:
df["weather_days_since_last_rain"] = df["weather_days_since_last_rain"].fillna(
    5)

Now we drop the columns related to the accident number and the address of the accident (given that now we have a complete list of Latitude and Longitude values).

Check for columns with null values

In [28]:
df.isna().sum().sum()

634

In [29]:
# VehicleType empty
df['vehicle_type'].value_counts(dropna=False)

vehicle_type
4.0    13301
3.0     3730
5.0      859
NaN      634
2.0      287
7.0      284
1.0      144
9.0       76
6.0       68
8.0       23
Name: count, dtype: int64

In [30]:
df["vehicle_type"] = (
    df["vehicle_type"]
    .fillna("unknown")
    .astype(int, errors="ignore")   # keep numbers, leave "unknown"
    .astype(str)                    # all as strings
    .astype("category")
)

In [31]:
df.isna().sum().sum()

0

Check for constant columns

In [32]:
# Columns that are constant (only 1 unique value)
const_cols = df.columns[df.nunique() == 1].tolist()
print("Constant columns:", const_cols)

# If you also want to see what that constant value is:
const_values = {col: df[col].iloc[0] for col in const_cols}
print("Constant column values:", const_values)

Constant columns: ['passengers_killed', 'weather_icy']
Constant column values: {'passengers_killed': 0, 'weather_icy': 0}


In [60]:
df.drop(['passengers_killed', 'weather_icy'], axis=1, inplace=True)

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19406 entries, 0 to 19405
Data columns (total 67 columns):
 #   Column                            Non-Null Count  Dtype                      
---  ------                            --------------  -----                      
 0   Protocollo                        19406 non-null  Int64                      
 1   DataOraIncidente                  19406 non-null  datetime64[ns, Europe/Rome]
 2   Gruppo                            19406 non-null  Int64                      
 3   Localizzazione1                   19406 non-null  object                     
 4   STRADA1                           19406 non-null  object                     
 5   Localizzazione2                   19406 non-null  object                     
 6   STRADA2                           19406 non-null  object                     
 7   Strada02                          19406 non-null  object                     
 8   Chilometrica                      19406 non-null  object

### CYCLICAL VARIABLES

YEAR - leave as numerical

MONTH - cyclical
DATE
DAY
TIME 


In [34]:
# Drop helper merge columns (won't error if already missing)
df = df.drop(columns=["time_local", "time_utc"], errors="ignore")

In [35]:
ts = df["DataOraIncidente"]

# Day of year (1..365/366) → numeric cyclic pair
df["dayofyear"] = ts.dt.dayofyear.astype("int16")
df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.0)
df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.0)

hours = ts.dt.hour
minutes = ts.dt.minute
df["time_float"] = hours + minutes / 60.0
df["time_sin"] = np.sin(2 * np.pi * df["time_float"] / 24.0)
df["time_cos"] = np.cos(2 * np.pi * df["time_float"] / 24.0)

# Treat MONTH and DAY as categorical (unordered) for k-prototypes
df["MONTH"] = df["MONTH"].astype("category")
df["DAY"] = df["DAY"].astype("category")   # 1=Mon ... 7=Sun as you defined

df = df.drop(columns=["time_float"], errors="ignore")

In [36]:
df.columns

Index(['Protocollo', 'DataOraIncidente', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'parked_vehicle_involved',
       'driver_injury', 'driver_gender', 'passenger_no_of_females',
       'passenger_no_of_males', 'passenger_no_of_unknown_sex',
       'passengers_killed', 'passengers_uninjured', 'passengers_injured',
       'phase_of_day', 'YEAR', 'MONTH', 'DATE', 'TIME', 'DAY',
       'pedestrian_gender', 'road_conditions', 'road_markings_absent',
       'road_markings_onroad', 'road_markings_signposts',
       'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type',
       'TipoStradaDifficulty', 'visibility', 'pedestrian_injury',
       'lighting_insufficient', 'traffic_density', 'vehicle_type',
       'vehicle_unknown', 'hit_and_run', 'DataOraIncidente_utc_rounded',
       'time', 'temperature_2m (°C)', 'relative_humi

In [37]:
# --- Columns to exclude from modeling (datetime/helpers/redundant) ---
drop_cols = [
    "DataOraIncidente_utc_rounded",  # helper
    "time",                          # helper datetime
    # raw time as object (you have time_sin/cos)
    "TIME",
    "DATE",                          # day-of-month; redundant with dayofyear
    "dayofyear"                      # redundant with doy_sin and doy_cos
]

df = df.drop(columns=drop_cols, errors="ignore")

In [38]:
df.to_parquet('010_data_cleaned.parquet', index=False)

In [ ]:
metadata = {
    'notebook': '011 Final data cleaning.ipynb',
    'step': 'final schema, feature selection, dtypes, QA, and exports',

    # Lineage (prefer geocoded if present)
    'input_sources': [
        {'path': '009_weather_included.csv', 'role': 'base'},
        {'path': '010i_geocoded_addresses.xlsx', 'role': 'geocoding_optional'},
        {'path': '010ii_gmaps_geocoded.parquet', 'role': 'geocoding_optional'}
    ],
    'output_files': {
        'parquet': '011_model_ready.parquet',
        'csv': '011_model_ready.csv',
        'features': '011_feature_lists.json',
        'metadata': '011_final_metadata.json'
    },

    # Primary key & uniqueness
    'primary_key': 'Protocollo',
    'rows_unique_by_pk': True,
    'duplicates_removed_pk': 0,

    # Column engineering in this step
    'columns_added': [
        # choose the one you used (keep both listed if you created both)
        'ped_count_binned',        # {1, 2, 3+} as category (if created)
        # in [0,1] from num_male/(num_male+num_female) (if created)
        'male_share'
    ],
    'columns_removed': [
        'total_injury_severity',   # dropped as redundant with winsorized severity
        'max_injury_severity',     # ditto
        'num_pedestrians_hit',     # replaced by multiple_pedestrians OR ped_count_binned
        'num_male', 'num_female'   # replaced by male_share OR removed to avoid demographic bias
    ],
    'columns_renamed': {},
    'columns_casted': {
        # enforce final dtypes
        'Protocollo': 'Int64',
        'severity_winsorized': 'float64',
        'multiple_pedestrians': 'int8',
        'ped_count_binned': 'category',
        'phase_of_day': 'category',
        'road_conditions': 'category',
        'road_surface': 'category',
        'accident_type': 'category',
        'visibility_ord': 'Int8',      # if present
        'road_conditions_ord': 'Int8',
        'road_surface_ord': 'Int8'
    },

    # Feature selection decisions (for k-prototypes and friends)
    'features_selected': {
        'numeric': [
            'severity_winsorized', 'time_sin', 'time_cos', 'doy_sin', 'doy_cos',
            'temperature_2m (°C)', 'relative_humidity_2m (%)',
            'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)',
            'precipitation (mm)', 'cloud_cover (%)'
            # + any other numerics you kept
        ],
        'categorical': [
            # choose one representation for group size:
            'multiple_pedestrians',             # OR replace with 'ped_count_binned'
            'phase_of_day', 'traffic_density', 'road_conditions',
            'road_surface', 'road_features', 'accident_type',
            'holiday_italy', 'is_weekend'
        ],
        'ordinal_codes': [
            # only if used downstream as numeric encodings
            'visibility_ord', 'road_conditions_ord', 'road_surface_ord', 'phase_of_day_ord'
        ],
        'id_columns': ['Protocollo'],
        'time_columns': ['DataOraIncidente', 'timestamp_local'],
        'geo_columns': ['Latitude', 'Longitude']
    },
    'features_excluded_high_corr': [
        # from your correlation review
        'total_injury_severity', 'max_injury_severity', 'num_pedestrians_hit',
        'num_male', 'num_female'
    ],
    'correlation_threshold': 0.85,

    # Filters applied here (if any)
    'filters_applied': [
        # add entries if you dropped rows for NA or invalids in this step
        # {'name':'drop_rows_missing_core_fields', 'before':(<r>,<c>), 'after':(<r>,<c>), 'rows_removed':<n>}
    ],

    # QA checks
    'qa_checks': {
        'pk_nulls': 0,
        'pk_unique': True,
        'lat_lon_missing_after': 0,
        'lat_bounds': [41.60, 42.20],
        'lon_bounds': [12.20, 12.80],
        'rows_outside_bbox': '<fill>',
        'severity_range_valid': True,   # 0–4 or winsorized within expected cap
        'na_rate_by_column_top': '<fill: dict of top NA columns>',
        'category_domain_valid': True,
        'correlation_max_abs': '<fill: e.g., 0.82 after drops>'
    },

    # Decisions & rationale
    'decisions_made': [
        'Retained severity_winsorized as the single severity signal; removed raw/max totals to avoid double weighting.',
        'Represented group size with a single feature (binary multiple_pedestrians or binned ped_count_binned).',
        'Removed gender counts to prevent redundancy and reduce demographic bias; optional male_share only if justified.',
        'Enforced final dtypes (ordered categoricals where appropriate) and validated domains/coordinate bounds.',
        'Produced explicit feature lists (numeric/categorical/ordinal) for k-prototypes and ablation tests.'
    ],

    # Modeling notes
    'modeling_notes': (
        "Re-tune gamma after feature changes. Consider RobustScaler for numerics with heavy tails "
        "before k-prototypes; keep ordinal columns as categories unless explicitly testing numeric codes."
    )}
}